In [1]:
"""
=================================================================================
OralVision - Ablation Study for EfficientNet-B0 + Coordinate Attention +
              Logistic Regression head (M4, merged dataset)
=================================================================================
Mirrors the M2 (CBAM) ablation script structure, adapted for M4's two
architectural changes:
  1. Attention: CBAM -> Coordinate Attention (factorized H/W attention)
  2. Classifier head: Linear+Dropout -> plain Logistic Regression (Linear only)

Also rebuilds the GROUP-AWARE split (orig__/beng__ pairs of the same photo
always land in the same split) since M4 was trained on the MERGED dataset,
not the bengram-only dataset M2 used - a plain stratified split here would
leak the orig/beng pair of a photo across train/test, same issue the m3/m4
training scripts already solved.
=================================================================================
"""

import os
import re
import copy
import json
import time
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, RandomSampler
from torchvision import transforms, models
from PIL import Image

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import precision_recall_fscore_support, accuracy_score

# =================================================================================
# CONFIG - edit these two paths for your machine
# =================================================================================
DATA_DIR = r"C:\Users\R Y Z E N\Desktop\oral_dataset_merged"
OUTPUT_DIR = r"C:\Users\R Y Z E N\Desktop\m4_coordatt_logreg-ab-study"
ABLATION_DIR = os.path.join(OUTPUT_DIR, "ablation")

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 16
NUM_WORKERS = 0
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
DIFFICULTY_ALPHA = 1.5

ABLATION_WARMUP_EPOCHS = 5
ABLATION_CASCADE_EPOCHS = 10
ABLATION_EPOCHS_SINGLE_PHASE = 15

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
np.random.seed(SEED)
os.makedirs(ABLATION_DIR, exist_ok=True)

RESULTS_JSON_PATH = os.path.join(ABLATION_DIR, "ablation_results_partial.json")


def log(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")


def safe_filename(name):
    return (name.replace(" ", "_").replace("(", "").replace(")", "")
                .replace(",", "").replace("=", "").replace("/", "-"))


# =================================================================================
# Data
# =================================================================================
def build_samples(data_dir):
    classes = sorted([d for d in os.listdir(data_dir)
                       if os.path.isdir(os.path.join(data_dir, d))])
    class_to_idx = {c: i for i, c in enumerate(classes)}
    samples = []
    for c in classes:
        folder = os.path.join(data_dir, c)
        for fname in os.listdir(folder):
            if fname.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
                samples.append((os.path.join(folder, fname), class_to_idx[c]))
    return samples, class_to_idx


def get_group_id(filepath):
    """
    Strips orig__/beng__ prefix so both versions of the same source photo
    share one group id - guarantees they land in the same split. Identical
    logic to the m3/m4 training scripts. Without this, a plain stratified
    split (like M2 used) would leak photo pairs across train/test.
    """
    folder = os.path.dirname(filepath)
    fname = os.path.basename(filepath)
    fname_stripped = re.sub(r"^(orig|beng)__", "", fname)
    return os.path.join(folder, fname_stripped)


class OralDataset(Dataset):
    def __init__(self, samples, indices, transform=None):
        self.samples = samples
        self.indices = indices
        self.transform = transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        gi = self.indices[i]
        path, label = self.samples[gi]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label, gi


# =================================================================================
# Model - Coordinate Attention (configurable for ablation)
# =================================================================================
class ConfigurableCoordAtt(nn.Module):
    """
    Factorized coordinate attention (Hou et al., 2021).
    attention_mode: 'full' | 'height_only' | 'width_only' | 'none'
      - 'full'        : standard CoordAtt, attends along both H and W
      - 'height_only' : only the height-direction attention map is applied
      - 'width_only'  : only the width-direction attention map is applied
      - 'none'        : module is skipped entirely (identity) - isolates the
                         contribution of attention altogether, same role as
                         M2 ablation's "No Attention" variant
    """
    def __init__(self, channels, reduction=32, attention_mode="full"):
        super().__init__()
        self.attention_mode = attention_mode
        self.pool_h = nn.AdaptiveAvgPool2d((None, 1))
        self.pool_w = nn.AdaptiveAvgPool2d((1, None))

        mip = max(8, channels // reduction)
        self.conv1 = nn.Conv2d(channels, mip, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(mip)
        self.act = nn.Hardswish()

        self.conv_h = nn.Conv2d(mip, channels, kernel_size=1, stride=1, padding=0)
        self.conv_w = nn.Conv2d(mip, channels, kernel_size=1, stride=1, padding=0)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        if self.attention_mode == "none":
            return x

        identity = x
        n, c, h, w = x.size()

        x_h = self.pool_h(x)                      # (n, c, h, 1)
        x_w = self.pool_w(x).permute(0, 1, 3, 2)   # (n, c, w, 1)

        y = torch.cat([x_h, x_w], dim=2)           # (n, c, h+w, 1)
        y = self.conv1(y)
        y = self.bn1(y)
        y = self.act(y)

        x_h, x_w = torch.split(y, [h, w], dim=2)
        x_w = x_w.permute(0, 1, 3, 2)               # back to (n, mip, 1, w)

        a_h = self.sigmoid(self.conv_h(x_h))        # (n, c, h, 1)
        a_w = self.sigmoid(self.conv_w(x_w))        # (n, c, 1, w)

        if self.attention_mode == "full":
            return identity * a_h * a_w
        elif self.attention_mode == "height_only":
            return identity * a_h
        elif self.attention_mode == "width_only":
            return identity * a_w
        else:
            raise ValueError(f"Unknown attention_mode: {self.attention_mode}")


class EfficientNetCoordAtt(nn.Module):
    """
    classifier_head: 'logreg' | 'linear_dropout'
      - 'logreg'        : plain Linear(1280 -> num_classes), no dropout - this
                           is what M4 actually used
      - 'linear_dropout': Dropout(dropout) + Linear(1280 -> num_classes) - the
                           heavier head M2/M3 used, included so the ablation
                           can isolate whether the simpler LogReg head helped
                           or hurt vs. the dropout head
    """
    def __init__(self, num_classes, ca_reduction=32, dropout=0.3,
                 attention_mode="full", classifier_head="logreg", pretrained=True):
        super().__init__()
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        base = models.efficientnet_b0(weights=weights)
        self.features = base.features
        self.coordatt = ConfigurableCoordAtt(1280, ca_reduction, attention_mode)
        self.avgpool = nn.AdaptiveAvgPool2d(1)

        if classifier_head == "logreg":
            self.classifier = nn.Linear(1280, num_classes)
        elif classifier_head == "linear_dropout":
            self.classifier = nn.Sequential(
                nn.Dropout(dropout),
                nn.Linear(1280, num_classes),
            )
        else:
            raise ValueError(f"Unknown classifier_head: {classifier_head}")

    def forward(self, x):
        x = self.features(x)
        x = self.coordatt(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


# =================================================================================
# Rebuild exact split (GROUP-AWARE - see get_group_id above)
# =================================================================================
log("Rebuilding dataset + GROUP-AWARE split (must match m4 training exactly)...")
samples, class_to_idx = build_samples(DATA_DIR)
idx_to_class = {v: k for k, v in class_to_idx.items()}
labels_all = [s[1] for s in samples]
num_classes = len(class_to_idx)
class_names = [idx_to_class[i] for i in range(num_classes)]

group_ids = [get_group_id(path) for path, _ in samples]
unique_groups = sorted(set(group_ids))
group_to_label = {}
for gid, lbl in zip(group_ids, labels_all):
    group_to_label[gid] = lbl
group_labels = [group_to_label[g] for g in unique_groups]

trainval_groups, test_groups = train_test_split(
    unique_groups, test_size=TEST_SPLIT, stratify=group_labels, random_state=SEED
)
val_frac_adj = VAL_SPLIT / (1 - TEST_SPLIT)
trainval_group_labels = [group_to_label[g] for g in trainval_groups]
train_groups, val_groups = train_test_split(
    trainval_groups, test_size=val_frac_adj, stratify=trainval_group_labels, random_state=SEED
)

train_groups_set, val_groups_set, test_groups_set = set(train_groups), set(val_groups), set(test_groups)
train_idx = [i for i, g in enumerate(group_ids) if g in train_groups_set]
val_idx = [i for i, g in enumerate(group_ids) if g in val_groups_set]
test_idx = [i for i, g in enumerate(group_ids) if g in test_groups_set]

overlap_tv = train_groups_set & val_groups_set
overlap_tt = train_groups_set & test_groups_set
overlap_vt = val_groups_set & test_groups_set
if overlap_tv or overlap_tt or overlap_vt:
    raise RuntimeError(f"LEAK DETECTED: overlapping groups across splits! "
                        f"train∩val={len(overlap_tv)}, train∩test={len(overlap_tt)}, "
                        f"val∩test={len(overlap_vt)}")
log(f"Split -> train: {len(train_idx)}, val: {len(val_idx)}, test: {len(test_idx)} "
    f"(unique photo groups: {len(train_groups_set)}/{len(val_groups_set)}/{len(test_groups_set)})")
log("Sanity check passed: no orig/beng pair of the same photo was split across train/val/test.")

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

AUG_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
NOAUG_TF = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
EVAL_TF = NOAUG_TF

val_ds = OralDataset(samples, val_idx, transform=EVAL_TF)
test_ds = OralDataset(samples, test_idx, transform=EVAL_TF)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

train_labels = [labels_all[i] for i in train_idx]
class_weights_arr = compute_class_weight(class_weight="balanced",
                                          classes=np.unique(train_labels), y=train_labels)
CLASS_WEIGHTS_BALANCED = {i: w for i, w in zip(np.unique(train_labels), class_weights_arr)}


# =================================================================================
# Sampler builder
# =================================================================================
def build_sampler(train_idx, labels_all, mode, difficulty_dict=None):
    if mode == "random":
        return RandomSampler(train_idx)

    weights = []
    for gi in train_idx:
        lbl = labels_all[gi]
        w = CLASS_WEIGHTS_BALANCED[lbl]
        if mode == "difficulty" and difficulty_dict is not None:
            diff = difficulty_dict.get(gi, 1.0)
            w = w * (diff ** DIFFICULTY_ALPHA)
        weights.append(w)
    weights = np.array(weights, dtype=np.float64)
    weights = weights / weights.sum() * len(weights)
    return WeightedRandomSampler(weights, num_samples=len(weights), replacement=True)


# =================================================================================
# Train / eval one epoch
# =================================================================================
def run_epoch(model, loader, class_weight_tensor, optimizer, difficulty_dict, train=True):
    model.train() if train else model.eval()
    total, correct, total_loss = 0, 0, 0.0
    all_preds, all_labels = [], []
    with torch.set_grad_enabled(train):
        for imgs, labels, idxs in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            if train:
                optimizer.zero_grad()
            outputs = model(imgs)
            per_sample_loss = nn.functional.cross_entropy(
                outputs, labels, weight=class_weight_tensor, reduction="none"
            )
            loss = per_sample_loss.mean()
            if train:
                loss.backward()
                optimizer.step()
                with torch.no_grad():
                    losses_np = per_sample_loss.detach().cpu().numpy()
                    for gi, l in zip(idxs.numpy(), losses_np):
                        prev = difficulty_dict.get(int(gi), 1.0)
                        difficulty_dict[int(gi)] = 0.5 * prev + 0.5 * float(l)
            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += imgs.size(0)
            all_preds.extend(preds.cpu().numpy().tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / total, correct / total, f1


@torch.no_grad()
def evaluate_test(model):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels, idxs in test_loader:
        imgs = imgs.to(DEVICE)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return acc, prec, rec, f1


# =================================================================================
# Variant definitions
# =================================================================================
VARIANTS = [
    dict(name="1. Baseline (Full Model)",
         attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),

    dict(name="2. No Attention (CoordAtt removed)",
         attention_mode="none", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),

    dict(name="3. Height-Only Attention",
         attention_mode="height_only", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),

    dict(name="4. Width-Only Attention",
         attention_mode="width_only", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),

    dict(name="5. No Curriculum (single-phase)",
         attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=False, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),

    dict(name="6. No Hard-Example Cascade",
         attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="class_only", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),

    dict(name="7. No Class Weighting",
         attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="random", class_weighted_loss=False,
         augmentation=True, lr_schedule=True),

    dict(name="8. Naive Training (no CoordAtt, no curriculum, no weighting)",
         attention_mode="none", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=False, sampler_mode="random", class_weighted_loss=False,
         augmentation=True, lr_schedule=True),

    dict(name="9. CoordAtt Reduction r=16",
         attention_mode="full", ca_reduction=16, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),

    dict(name="10. CoordAtt Reduction r=8",
         attention_mode="full", ca_reduction=8, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),

    dict(name="11. No Data Augmentation",
         attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=False, lr_schedule=True),

    dict(name="12. Fixed LR (no ReduceLROnPlateau)",
         attention_mode="full", ca_reduction=32, classifier_head="logreg", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=False),

    dict(name="13. Linear+Dropout Head (instead of Logistic Regression)",
         attention_mode="full", ca_reduction=32, classifier_head="linear_dropout", dropout=0.3,
         curriculum=True, sampler_mode="difficulty", class_weighted_loss=True,
         augmentation=True, lr_schedule=True),
]


def train_variant(cfg):
    train_tf = AUG_TF if cfg["augmentation"] else NOAUG_TF
    train_ds = OralDataset(samples, train_idx, transform=train_tf)

    model = EfficientNetCoordAtt(
        num_classes=num_classes,
        ca_reduction=cfg["ca_reduction"],
        dropout=cfg["dropout"],
        attention_mode=cfg["attention_mode"],
        classifier_head=cfg["classifier_head"],
    ).to(DEVICE)

    class_weight_tensor = None
    if cfg["class_weighted_loss"]:
        class_weight_tensor = torch.tensor(
            [CLASS_WEIGHTS_BALANCED[i] for i in range(num_classes)], dtype=torch.float
        ).to(DEVICE)

    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = None
    if cfg["lr_schedule"]:
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="min", factor=0.5, patience=3, min_lr=1e-7
        )

    difficulty_dict = {}
    best_f1, best_state = -1.0, None

    total_epochs = ABLATION_EPOCHS_SINGLE_PHASE if not cfg["curriculum"] else \
        (ABLATION_WARMUP_EPOCHS + ABLATION_CASCADE_EPOCHS)

    t0 = time.time()
    for epoch in range(1, total_epochs + 1):
        if cfg["curriculum"]:
            use_difficulty = (epoch > ABLATION_WARMUP_EPOCHS) and (cfg["sampler_mode"] == "difficulty")
        else:
            use_difficulty = False

        effective_mode = cfg["sampler_mode"]
        if effective_mode == "difficulty" and not use_difficulty:
            sampler = build_sampler(train_idx, labels_all, "class_only")
        else:
            sampler = build_sampler(train_idx, labels_all, effective_mode, difficulty_dict)

        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS)

        train_loss, train_acc, train_f1 = run_epoch(
            model, train_loader, class_weight_tensor, optimizer, difficulty_dict, train=True
        )
        val_loss, val_acc, val_f1 = run_epoch(
            model, val_loader, class_weight_tensor, optimizer, difficulty_dict, train=False
        )
        if scheduler is not None:
            scheduler.step(val_loss)

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = copy.deepcopy(model.state_dict())

        log(f"  epoch {epoch:02d}/{total_epochs} | train_loss {train_loss:.4f} | "
            f"val_f1 {val_f1:.4f} (best {best_f1:.4f})")

    runtime = time.time() - t0
    model.load_state_dict(best_state)
    test_acc, test_prec, test_rec, test_f1 = evaluate_test(model)
    return {
        "accuracy": test_acc, "precision": test_prec, "recall": test_rec, "f1": test_f1,
        "runtime_sec": runtime,
    }, model


def train_variant_with_retry(cfg, max_retries=1):
    for attempt in range(max_retries + 1):
        try:
            return train_variant(cfg)
        except RuntimeError as e:
            if "CUDA" in str(e) and attempt < max_retries:
                log(f"  !! CUDA error on variant '{cfg['name']}' (attempt {attempt+1}): {e}")
                log("  !! Clearing CUDA cache and retrying this variant once...")
                torch.cuda.empty_cache()
                time.sleep(10)
                continue
            raise


# =================================================================================
# Robustness checks
# =================================================================================
@torch.no_grad()
def evaluate_with_gaussian_noise(model, sigma=0.05):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels, idxs in test_loader:
        imgs = imgs.to(DEVICE)
        noise = torch.randn_like(imgs) * sigma
        outputs = model(imgs + noise)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return acc, f1


def evaluate_with_fgsm(model, epsilon=0.03):
    model.eval()
    all_preds, all_labels = [], []
    for imgs, labels, idxs in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        imgs.requires_grad_(True)
        outputs = model(imgs)
        loss = nn.functional.cross_entropy(outputs, labels)
        model.zero_grad()
        loss.backward()
        perturbed = imgs + epsilon * imgs.grad.sign()
        with torch.no_grad():
            outputs_adv = model(perturbed)
            preds = outputs_adv.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds.tolist())
        all_labels.extend(labels.cpu().numpy().tolist())
    acc = accuracy_score(all_labels, all_preds)
    _, _, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return acc, f1


# =================================================================================
# Main - with resume support and incremental saving
# =================================================================================
def main():
    results = {}
    baseline_model = None
    baseline_model_acc = None

    if os.path.exists(RESULTS_JSON_PATH):
        with open(RESULTS_JSON_PATH, "r", encoding="utf-8") as f:
            results = json.load(f)
        log(f"Resuming — {len(results)} variant(s) already completed: {list(results.keys())}")

    for cfg in VARIANTS:
        if cfg["name"] in results:
            log(f"Skipping already-completed variant: {cfg['name']}")
            if cfg["name"].startswith("1."):
                baseline_model = EfficientNetCoordAtt(
                    num_classes=num_classes,
                    ca_reduction=cfg["ca_reduction"],
                    dropout=cfg["dropout"],
                    attention_mode=cfg["attention_mode"],
                    classifier_head=cfg["classifier_head"],
                ).to(DEVICE)
                baseline_model.load_state_dict(
                    torch.load(os.path.join(ABLATION_DIR, "baseline_ablation_model.pth"), map_location=DEVICE)
                )
                baseline_model_acc = results[cfg["name"]]["accuracy"]
            continue

        log("=" * 70)
        log(f"Training variant: {cfg['name']}")
        log("=" * 70)
        metrics, model = train_variant_with_retry(cfg)
        results[cfg["name"]] = metrics
        log(f"  -> Test: acc={metrics['accuracy']:.4f} macro_f1={metrics['f1']:.4f} "
            f"runtime={metrics['runtime_sec']:.1f}s")

        with open(RESULTS_JSON_PATH, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2)

        torch.save(model.state_dict(),
                   os.path.join(ABLATION_DIR, f"model_{safe_filename(cfg['name'])}.pth"))

        if cfg["name"].startswith("1."):
            baseline_model = model
            baseline_model_acc = metrics["accuracy"]
            torch.save(model.state_dict(), os.path.join(ABLATION_DIR, "baseline_ablation_model.pth"))

    baseline_f1 = results["1. Baseline (Full Model)"]["f1"]

    log("=" * 70)
    log("Robustness checks (baseline model, no retraining)")
    log("=" * 70)
    noise_acc, noise_f1 = evaluate_with_gaussian_noise(baseline_model, sigma=0.05)
    log(f"  Gaussian noise (sigma=0.05): acc={noise_acc:.4f} macro_f1={noise_f1:.4f}")
    fgsm_acc, fgsm_f1 = evaluate_with_fgsm(baseline_model, epsilon=0.03)
    log(f"  FGSM (epsilon=0.03): acc={fgsm_acc:.4f} macro_f1={fgsm_f1:.4f}")

    lines = [
        "OralVision - Ablation Study Results (M4 - Coordinate Attention + Logistic Regression)",
        "=" * 80,
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
        f"Ablation training budget: curriculum variants = "
        f"{ABLATION_WARMUP_EPOCHS}+{ABLATION_CASCADE_EPOCHS} epochs, "
        f"single-phase variants = {ABLATION_EPOCHS_SINGLE_PHASE} epochs",
        "",
        f"{'Variant':55s} {'Acc':>8s} {'Macro-Prec':>11s} {'Macro-Rec':>10s} {'Macro-F1':>9s} "
        f"{'Delta F1 vs base':>17s} {'Runtime(s)':>11s}",
        "-" * 125,
    ]
    for name, m in results.items():
        delta = baseline_f1 - m["f1"]
        lines.append(
            f"{name:55s} {m['accuracy']:>8.4f} {m['precision']:>11.4f} {m['recall']:>10.4f} "
            f"{m['f1']:>9.4f} {delta:>17.4f} {m['runtime_sec']:>11.1f}"
        )
    lines += [
        "-" * 125,
        "",
        "Robustness (baseline model only, no retraining):",
        f"  Clean test:                   Acc={baseline_model_acc:.4f}  Macro-F1={baseline_f1:.4f}",
        f"  Gaussian noise (sigma=0.05):  Acc={noise_acc:.4f}  Macro-F1={noise_f1:.4f}  "
        f"(Delta F1={baseline_f1 - noise_f1:.4f})",
        f"  FGSM attack (epsilon=0.03):   Acc={fgsm_acc:.4f}  Macro-F1={fgsm_f1:.4f}  "
        f"(Delta F1={baseline_f1 - fgsm_f1:.4f})",
        "",
        "Note: Precision, Recall, and F1 above are macro-averaged across all "
        f"{num_classes} classes ({', '.join(class_names)}). "
        "Accuracy is overall (micro) accuracy across all test samples.",
        "",
        "Delta F1 = F1_baseline - F1_variant (positive = component/setting helps; "
        "negative = variant outperformed baseline, worth investigating further).",
        "",
        "Split strategy: GROUP-AWARE (orig/beng pair of same photo always kept in the "
        "same split - see get_group_id). This matches the m4 training script exactly.",
    ]
    summary_txt = "\n".join(lines)

    with open(os.path.join(ABLATION_DIR, "ablation_results.txt"), "w", encoding="utf-8") as f:
        f.write(summary_txt)
    log("Saved ablation_results.txt")
    print("\n" + summary_txt + "\n")

    with open(os.path.join(ABLATION_DIR, "ablation_results.json"), "w", encoding="utf-8") as f:
        json.dump({
            "variants": results,
            "robustness": {
                "clean": {"acc": baseline_model_acc, "f1": baseline_f1},
                "gaussian_noise": {"acc": noise_acc, "f1": noise_f1},
                "fgsm": {"acc": fgsm_acc, "f1": fgsm_f1},
            },
        }, f, indent=2)
    log("Saved ablation_results.json")

    names_short = [n.split(".")[0] + "." for n in results.keys()]
    accs = [m["accuracy"] for m in results.values()]
    f1s = [m["f1"] for m in results.values()]

    x = np.arange(len(names_short))
    width = 0.35
    plt.figure(figsize=(14, 6))
    plt.bar(x - width / 2, accs, width, label="Accuracy")
    plt.bar(x + width / 2, f1s, width, label="Macro F1")
    plt.axhline(baseline_f1, color="gray", linestyle="--", linewidth=1, label="Baseline Macro F1")
    plt.xticks(x, names_short, rotation=0)
    plt.ylim(0, 1.05)
    plt.ylabel("Score")
    plt.title("OralVision Ablation Study (M4) - Accuracy & Macro F1 by Variant")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(ABLATION_DIR, "ablation_summary.png"), dpi=150)
    plt.close()
    log("Saved ablation_summary.png")

    log("=" * 70)
    log(f"All ablation outputs saved to: {ABLATION_DIR}")
    log("=" * 70)


if __name__ == "__main__":
    main()

[09:04:00] Rebuilding dataset + GROUP-AWARE split (must match m4 training exactly)...
[09:04:04] Split -> train: 22212, val: 4760, test: 4760 (unique photo groups: 11106/2380/2380)
[09:04:04] Sanity check passed: no orig/beng pair of the same photo was split across train/val/test.
[09:04:04] Resuming — 12 variant(s) already completed: ['1. Baseline (Full Model)', '2. No Attention (CoordAtt removed)', '3. Height-Only Attention', '4. Width-Only Attention', '5. No Curriculum (single-phase)', '6. No Hard-Example Cascade', '7. No Class Weighting', '8. Naive Training (no CoordAtt, no curriculum, no weighting)', '9. CoordAtt Reduction r=16', '10. CoordAtt Reduction r=8', '11. No Data Augmentation', '12. Fixed LR (no ReduceLROnPlateau)']
[09:04:04] Skipping already-completed variant: 1. Baseline (Full Model)


C:\Users\R Y Z E N\AppData\Local\Temp\ipykernel_32796\2299666616.py:584: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(os.path.join(ABLATION_DIR, "baseline_ablati

[09:04:05] Skipping already-completed variant: 2. No Attention (CoordAtt removed)
[09:04:05] Skipping already-completed variant: 3. Height-Only Attention
[09:04:05] Skipping already-completed variant: 4. Width-Only Attention
[09:04:05] Skipping already-completed variant: 5. No Curriculum (single-phase)
[09:04:05] Skipping already-completed variant: 6. No Hard-Example Cascade
[09:04:05] Skipping already-completed variant: 7. No Class Weighting
[09:04:05] Skipping already-completed variant: 8. Naive Training (no CoordAtt, no curriculum, no weighting)
[09:04:05] Skipping already-completed variant: 9. CoordAtt Reduction r=16
[09:04:05] Skipping already-completed variant: 10. CoordAtt Reduction r=8
[09:04:05] Skipping already-completed variant: 11. No Data Augmentation
[09:04:05] Skipping already-completed variant: 12. Fixed LR (no ReduceLROnPlateau)
[09:04:05] ======================================================================
[09:04:05] Training variant: 13. Linear+Dropout Head (instea